# Revision 10D — Temporal matched retained fractions through K=50

**Recommended repository location**

```text
analysis/03_core_results/
revision_10d_temporal_matched_fraction_K5_K50.ipynb
```

This focused follow-on notebook answers one question:

> Does the apparent fixed-top-10 advantage near temporal model order K=14
> persist when the number of selected archetypes is matched as a fraction of K?

It evaluates the canonical temporal MS-AA fits for intact and word-scrambled
listening at:

```text
K = 5, 8, 10, 12, 14, 16, 20, 25, 30, 40, 50
```

and retained fractions:

```text
25%, 50%, 70%, and 100%
```

The notebook is deliberately limited to the informative stimulus conditions and
to K <= 50 for the first pass. It:

- reuses existing participant-held-out archetype rankings whenever compatible;
- reuses exact existing (K, m) reconstruction-space evaluations;
- treats stored full-model evaluations as the 100% condition;
- computes only missing cells;
- checkpoints after every outer-split direction;
- writes into the existing Revision 10 output directory so prior cached work is
  preserved.

The generator-anchor masking analysis is not run here.


In [ ]:
# ============================================================
# USER SETTINGS
# ============================================================

from pathlib import Path

ANALYSIS_TYPE = "temporal"
FIT_SCOPE = "across"
REFERENCE_K = 14
CONDITIONS = ["intact", "word"]
CONDITION_INDEX = {condition: i for i, condition in enumerate(CONDITIONS)}

# Reuse the Revision 10A output/cache directory.
OUTPUT_DIR = Path(
    "outputs/temporal_compact_dictionary_K14_local"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CANONICAL_FIT_DIR = Path("msaa_flexible_outputs_npz")
FIT_SEARCH_ROOTS = [CANONICAL_FIT_DIR]
EXPLICIT_K14_FITS = []
AUTO_DISCOVER_K14_SEEDS = False
K14_FIT_INDICES_TO_RUN = [0]

LEGACY_OUTPUT_DIR_CANDIDATES = [
    Path("msaa_heldout_topm_outputs_temporal_across_fullgrid"),
    Path("msaa_heldout_topm_outputs_temporal_across"),
]

REUSE_EXISTING_RANKINGS = True
REUSE_EXISTING_RECONSTRUCTION_RUNS = True

N_OUTER_REPS = 20
NFOLDS_DECODE = 2
NREPS_DECODE = 20
RNG_SEED = 42
EPS = 1e-8
SPLIT_SCHEME = "legacy_K_specific"

# ------------------------------------------------------------
# Analysis 1: matched retained fractions across K
# ------------------------------------------------------------

RUN_MATCHED_FRACTION = True
MATCHED_FRACTIONS = [0.25, 0.50, 0.70, 1.00]
MATCHED_FRACTION_K_VALUES = [
    5, 8, 10, 12, 14, 16, 20, 25, 30, 40, 50,
]
MATCHED_FRACTION_ALL_DISCOVERED_SEEDS = False

# ------------------------------------------------------------
# Analysis 2: generator-anchor-masked K=14 decoding
# ------------------------------------------------------------

RUN_ANCHOR_MASKED = False

# Primary control: remove each archetype's maximal-C generator TR plus/minus
# this many TRs. Use [0, 2] for an exact-anchor and broader-window sensitivity;
# use [2] for the single primary analysis and a shorter run.
ANCHOR_HALF_WINDOWS = [2]

# Evaluate the complete accumulation curve. A smaller diagnostic grid can be
# used for a smoke test, e.g. [1, 3, 5, 7, 10, 14].
ANCHOR_MASK_TOP_M_VALUES = list(range(1, REFERENCE_K + 1))

# Freeze the original unmasked cross-fitted ranking. This asks whether the
# same selected subsets identify non-anchor timepoints in held-out subjects.
ANCHOR_RANKING_MODE = "frozen_unmasked_crossfit"

# Existing unmasked K=14 runs created by Revision 10A.
UNMASKED_K14_RUN_CSV = (
    OUTPUT_DIR / "K14_accumulation_runs.csv"
)

SKIP_EXISTING = True
FORCE_RECOMPUTE = False

print("Output:", OUTPUT_DIR.resolve())
print("Matched fractions:", RUN_MATCHED_FRACTION)
print("Anchor-masked control:", RUN_ANCHOR_MASKED)
print("Matched-fraction K values:", MATCHED_FRACTION_K_VALUES)
print("Conditions:", CONDITIONS)


In [ ]:
# ============================================================
# IMPORTS / SMALL HELPERS
# ============================================================

import gc
import hashlib
import re
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from IPython.display import display

REPO = Path.cwd().resolve()
sys.path.insert(0, str(REPO / "src"))


def sem(values):
    values = pd.Series(values).dropna()
    if len(values) <= 1:
        return np.nan
    return float(values.std(ddof=1) / np.sqrt(len(values)))


def load_csv(path):
    path = Path(path)
    return pd.read_csv(path) if path.exists() and path.stat().st_size > 0 else pd.DataFrame()


def save_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def append_dedupe(base, new, keys):
    if new is None or len(new) == 0:
        return base
    out = new.copy() if base is None or len(base) == 0 else pd.concat([base, new], ignore_index=True)
    return out.drop_duplicates(keys, keep="last").reset_index(drop=True)


## 1. Discover and load temporal MS-AA fits

Copied fits are deduplicated by a fingerprint of the shared generator matrix
\(C\). The canonical repository fit is listed first when available.


In [ ]:
# ============================================================
# FIT DISCOVERY / LOADING
# ============================================================

FIT_NAME_TEMPLATE = "temporalAA_across_acrossCond_K{K}.npz"


def normalize_labels(values):
    out = []
    for value in np.asarray(values).reshape(-1):
        out.append(value.decode("utf-8") if isinstance(value, bytes) else str(value))
    return np.asarray(out, dtype=object)


def load_fit(path):
    path = Path(path)
    data = np.load(path, allow_pickle=True)
    results_subj = data["results_subj"].tolist()
    if isinstance(results_subj, np.ndarray):
        results_subj = results_subj.tolist()
    return {
        "path": path,
        "K": int(np.asarray(data["K"]).item()),
        "results_subj": results_subj,
        "condition_labels_str": normalize_labels(data["condition_labels_str"]),
        "C": np.asarray(data["C"], dtype=float) if "C" in data else None,
    }


def fit_fingerprint(path):
    fit = load_fit(path)
    digest = hashlib.sha1()
    if fit["C"] is not None:
        digest.update(np.round(fit["C"], 10).tobytes())
    else:
        digest.update(np.round(np.asarray(fit["results_subj"][0]["sXC"], dtype=float), 10).tobytes())
    return digest.hexdigest()[:16]


def seed_label_from_path(path, fallback_index):
    match = re.search(r"seed[_-]?(\d+)", str(path), flags=re.IGNORECASE)
    if match:
        return f"seed{int(match.group(1))}"
    canonical = CANONICAL_FIT_DIR / FIT_NAME_TEMPLATE.format(K=REFERENCE_K)
    try:
        if Path(path).resolve() == canonical.resolve():
            return "seed0"
    except Exception:
        pass
    return f"fit{fallback_index:02d}"


def discover_fit_paths(K):
    filename = FIT_NAME_TEMPLATE.format(K=int(K))
    candidates = []
    canonical = CANONICAL_FIT_DIR / filename
    if canonical.exists():
        candidates.append(canonical)
    if int(K) == REFERENCE_K:
        candidates.extend(Path(p) for p in EXPLICIT_K14_FITS if Path(p).exists())
    if AUTO_DISCOVER_K14_SEEDS or int(K) != REFERENCE_K:
        for root in FIT_SEARCH_ROOTS:
            root = Path(root)
            if not root.exists():
                continue
            if root.is_file():
                if root.name == filename:
                    candidates.append(root)
                continue
            direct = root / filename
            if direct.exists():
                candidates.append(direct)
            try:
                candidates.extend(root.glob(f"**/{filename}"))
            except OSError:
                pass

    rows, seen_paths, seen_fingerprints = [], set(), set()
    for path in candidates:
        try:
            resolved = path.resolve()
        except Exception:
            resolved = path
        if resolved in seen_paths:
            continue
        seen_paths.add(resolved)
        try:
            fit = load_fit(path)
            if fit["K"] != int(K):
                continue
            fp = fit_fingerprint(path)
        except Exception as exc:
            warnings.warn(f"Skipping unreadable fit {path}: {exc}")
            continue
        if fp in seen_fingerprints:
            continue
        seen_fingerprints.add(fp)
        rows.append({
            "fit_path": str(path),
            "fit_label": seed_label_from_path(path, len(rows)),
            "fingerprint": fp,
            "K": int(K),
            "n_observations": len(fit["results_subj"]),
        })

    table = pd.DataFrame(rows)
    if len(table):
        table["_canonical"] = table["fit_path"].astype(str).eq(str(canonical)).astype(int)
        table = table.sort_values(["_canonical", "fit_label"], ascending=[False, True]).drop(columns="_canonical").reset_index(drop=True)
    return table


def condition_subjects_from_fit(fit, condition):
    idx = np.where(fit["condition_labels_str"] == str(condition))[0]
    return [fit["results_subj"][int(i)] for i in idx]


k14_fit_table = discover_fit_paths(REFERENCE_K)
if len(k14_fit_table) == 0:
    raise FileNotFoundError("No K=14 temporal across-condition fits found. Add paths to EXPLICIT_K14_FITS or FIT_SEARCH_ROOTS.")

display(k14_fit_table)
K14_FIT_INDICES = list(range(len(k14_fit_table))) if K14_FIT_INDICES_TO_RUN is None else [int(i) for i in K14_FIT_INDICES_TO_RUN if 0 <= int(i) < len(k14_fit_table)]
print("K=14 fit indices:", K14_FIT_INDICES)


## 2. Optimized temporal held-out decoder

This adapts Ben's low-memory temporal cross-fit implementation. Single-archetype
ranking uses an exact low-rank correlation calculation, while cumulative
held-out reconstructions are built incrementally.


In [ ]:
# ============================================================
# DECODING CORE
# ============================================================


def get_xval_assignments(ndata, nfolds, rng):
    assignments = np.zeros(ndata, dtype=int)
    groupsize = int(np.ceil(ndata / nfolds))
    for fold in range(1, nfolds):
        inds = np.arange(fold * groupsize, min((fold + 1) * groupsize, ndata))
        assignments[inds] = fold
    rng.shuffle(assignments)
    return assignments


def decoder_metrics(corrs):
    T = corrs.shape[0]
    true_t = np.arange(T)
    pred = np.argmax(corrs, axis=1)
    return {
        "accuracy": float(np.mean(pred == true_t)),
        "error": float(np.mean(np.abs(pred - true_t)) / T),
        "rank": float(np.mean(np.mean(corrs <= np.diag(corrs)[:, None], axis=1))),
    }


def summarize_metric_rows(rows):
    df = pd.DataFrame(rows)
    return {
        "mean": float(df["accuracy"].mean()),
        "err": float(df["accuracy"].std(ddof=1) / np.sqrt(max(len(df), 1))),
        "std": float(df["accuracy"].std(ddof=1)),
        "count": int(len(df)),
        "rank_mean": float(df["rank"].mean()),
        "error_mean": float(df["error"].mean()),
    }


def inner_masks(n_subjects, nreps, seed):
    if NFOLDS_DECODE != 2:
        raise ValueError("Optimized decoder assumes NFOLDS_DECODE=2.")
    rng = np.random.default_rng(int(seed))
    masks = np.zeros((nreps, n_subjects), dtype=bool)
    for rep in range(nreps):
        masks[rep] = get_xval_assignments(n_subjects, 2, rng) == 0
    return masks


def batch_decode_stack(stack, nreps=20, seed=42):
    X = np.asarray(stack, dtype=float)
    N, T, F = X.shape
    if F < 2:
        return {"mean":np.nan,"err":np.nan,"std":np.nan,"count":0,"rank_mean":np.nan,"error_mean":np.nan}
    masks0 = inner_masks(N, nreps, seed)
    n0 = masks0.sum(axis=1).astype(float)
    n1 = (N - masks0.sum(axis=1)).astype(float)
    A = np.einsum("rn,ntf->rtf", masks0 / n0[:, None], X, optimize=True)
    B = np.einsum("rn,ntf->rtf", (~masks0) / n1[:, None], X, optimize=True)
    A = A - A.mean(axis=2, keepdims=True)
    B = B - B.mean(axis=2, keepdims=True)
    normA = np.linalg.norm(A, axis=2)
    normB = np.linalg.norm(B, axis=2)
    A = A / np.where(normA[:, :, None] > 0, normA[:, :, None], 1.0)
    B = B / np.where(normB[:, :, None] > 0, normB[:, :, None], 1.0)
    corrs = np.matmul(A, np.swapaxes(B, 1, 2))
    corrs = np.nan_to_num(corrs, nan=0.0, posinf=0.0, neginf=0.0)
    rows=[]
    for rep in range(nreps):
        m0=decoder_metrics(corrs[rep]); m0.update({"rep":rep,"fold":0}); rows.append(m0)
        m1=decoder_metrics(corrs[rep].T); m1.update({"rep":rep,"fold":1}); rows.append(m1)
    return summarize_metric_rows(rows)


def temporal_single_factors(subjects, archetype):
    U, W = [], []
    for sub in subjects:
        sXC = np.asarray(sub["sXC"], dtype=float)
        S = np.asarray(sub["S"], dtype=float)
        temporal = S[int(archetype), :]
        spatial = sXC[:, int(archetype)]
        centered_t = temporal - temporal.mean()
        sd_t = temporal.std()
        U.append(centered_t)
        W.append(spatial / (np.abs(spatial) * sd_t + EPS))
    U = np.asarray(U, dtype=float)
    W = np.asarray(W, dtype=float)
    Wc = W - W.mean(axis=1, keepdims=True)
    return U, Wc @ Wc.T


def lowrank_single_decode_summary(U, G, subset_indices, nreps, seed):
    subset_indices = np.asarray(subset_indices, dtype=int)
    rng = np.random.default_rng(int(seed))
    rows=[]
    for rep in range(nreps):
        fold_ids = get_xval_assignments(len(subset_indices), 2, rng)
        ia = subset_indices[fold_ids == 0]
        ib = subset_indices[fold_ids == 1]
        Ua, Ub = U[ia].T, U[ib].T
        Gaa = G[np.ix_(ia, ia)] / (len(ia) ** 2)
        Gbb = G[np.ix_(ib, ib)] / (len(ib) ** 2)
        Gab = G[np.ix_(ia, ib)] / (len(ia) * len(ib))
        numerator = Ua @ Gab @ Ub.T
        na = np.sqrt(np.maximum(np.einsum("ti,ij,tj->t", Ua, Gaa, Ua, optimize=True), 0.0))
        nb = np.sqrt(np.maximum(np.einsum("ti,ij,tj->t", Ub, Gbb, Ub, optimize=True), 0.0))
        denom = na[:, None] * nb[None, :]
        corrs = np.divide(numerator, denom, out=np.zeros_like(numerator), where=denom > 0)
        corrs = np.nan_to_num(corrs, nan=0.0, posinf=0.0, neginf=0.0)
        m0=decoder_metrics(corrs); m0.update({"rep":rep,"fold":0}); rows.append(m0)
        m1=decoder_metrics(corrs.T); m1.update({"rep":rep,"fold":1}); rows.append(m1)
    return summarize_metric_rows(rows)


In [ ]:
# ============================================================
# RECONSTRUCTION / EXPRESSION BUILDERS
# ============================================================


def temporal_contribution_stack(subjects, archetype):
    tvec = np.stack([np.asarray(sub["S"], dtype=float)[int(archetype), :] for sub in subjects], axis=0)
    vvec = np.stack([np.asarray(sub["sXC"], dtype=float)[:, int(archetype)] for sub in subjects], axis=0)
    return tvec[:, :, None] * vvec[:, None, :]


def normalize_stack_over_time(raw):
    return (raw - raw.mean(axis=1, keepdims=True)) / (raw.std(axis=1, keepdims=True) + EPS)


def full_temporal_reconstruction_stack(subjects):
    out=[]
    for sub in subjects:
        X = (np.asarray(sub["sXC"], dtype=float) @ np.asarray(sub["S"], dtype=float)).T
        out.append((X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + EPS))
    return np.stack(out, axis=0)


def temporal_expression_stack(subjects, indices=None):
    out=[]
    for sub in subjects:
        S=np.asarray(sub["S"], dtype=float)
        idx=np.arange(S.shape[0]) if indices is None else np.asarray(list(indices), dtype=int)
        X=S[idx, :].T
        out.append((X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + EPS))
    return np.stack(out, axis=0)


def evaluate_reconstruction_topm_fast(subjects, ranked, top_m_values, nreps, seed):
    K=len(ranked)
    desired=sorted({int(m) for m in top_m_values if 1 <= int(m) <= K})
    if not desired:
        return pd.DataFrame()
    full=full_temporal_reconstruction_stack(subjects)
    full_summary=batch_decode_stack(full, nreps=nreps, seed=seed)
    T,V=full.shape[1],full.shape[2]
    chance=1.0/T
    raw=np.zeros((len(subjects),T,V), dtype=float)
    rows=[]
    for pos,arch in enumerate(ranked[:max(desired)], start=1):
        raw += temporal_contribution_stack(subjects, arch)
        if pos not in desired:
            continue
        summary=batch_decode_stack(normalize_stack_over_time(raw), nreps=nreps, seed=seed)
        denom=full_summary["mean"]-chance
        retention=(summary["mean"]-chance)/denom if abs(denom)>1e-12 else np.nan
        rows.append({
            "space":"reconstruction","top_m":pos,"selected_archetypes":str([int(x) for x in ranked[:pos]]),
            "mean":summary["mean"],"err":summary["err"],"std":summary["std"],"count":summary["count"],
            "rank_mean":summary["rank_mean"],"error_mean":summary["error_mean"],
            "full_mean_same_eval":full_summary["mean"],"full_err_same_eval":full_summary["err"],
            "retention_above_chance":retention,"chance":chance,
        })
    return pd.DataFrame(rows)


def evaluate_expression_topm_fast(subjects, ranked, top_m_values, nreps, seed):
    K=len(ranked)
    desired=sorted({int(m) for m in top_m_values if 1 <= int(m) <= K})
    if not desired:
        return pd.DataFrame()
    full=temporal_expression_stack(subjects, None)
    full_summary=batch_decode_stack(full, nreps=nreps, seed=seed)
    chance=1.0/full.shape[1]
    rows=[]
    for m in desired:
        selected=ranked[:m]
        summary=(batch_decode_stack(temporal_expression_stack(subjects, selected), nreps=nreps, seed=seed)
                 if m >= 2 else {"mean":np.nan,"err":np.nan,"std":np.nan,"count":0,"rank_mean":np.nan,"error_mean":np.nan})
        denom=full_summary["mean"]-chance
        retention=((summary["mean"]-chance)/denom if np.isfinite(summary["mean"]) and abs(denom)>1e-12 else np.nan)
        rows.append({
            "space":"expression","top_m":m,"selected_archetypes":str([int(x) for x in selected]),
            "mean":summary["mean"],"err":summary["err"],"std":summary["std"],"count":summary["count"],
            "rank_mean":summary["rank_mean"],"error_mean":summary["error_mean"],
            "full_mean_same_eval":full_summary["mean"],"full_err_same_eval":full_summary["err"],
            "retention_above_chance":retention,"chance":chance,
        })
    return pd.DataFrame(rows)


## 3. Fixed outer splits and reusable rankings

The default split scheme reuses the same participant splits across model orders
and fit seeds, making the matched-fraction comparison paired with respect to
participant assignment.


In [ ]:
# ============================================================
# LEGACY SPLITS / SEEDS / CACHE IMPORT
# ============================================================

def outer_seed_for(condition, outer_rep, K):
    # Exact split definition used by the optimized temporal full-grid notebook.
    sequence = np.random.SeedSequence([
        int(RNG_SEED),
        1,  # temporal-analysis tag
        int(K),
        int(CONDITION_INDEX[condition]),
        int(outer_rep),
    ])
    return int(sequence.generate_state(1, dtype=np.uint32)[0])


def ranking_seed_for(K, outer_rep, direction_offset):
    return int(
        RNG_SEED
        + 100000 * int(outer_rep)
        + 1000 * int(K)
        + int(direction_offset)
    )


def evaluation_seed_for(K, outer_rep, direction_offset):
    return int(
        RNG_SEED
        + 200000 * int(outer_rep)
        + 1000 * int(K)
        + int(direction_offset)
    )


def split_outer_subject_indices(n_subjects, seed):
    rng = np.random.default_rng(int(seed))
    indices = np.arange(n_subjects)
    rng.shuffle(indices)
    midpoint = n_subjects // 2
    return indices[:midpoint], indices[midpoint:]


def direction_definitions(n_subjects, condition, K):
    rows = []
    for outer_rep in range(N_OUTER_REPS):
        outer_seed = outer_seed_for(condition, outer_rep, K)
        group_a, group_b = split_outer_subject_indices(
            n_subjects,
            outer_seed,
        )
        rows.extend([
            {
                "outer_rep": outer_rep,
                "outer_seed": outer_seed,
                "direction": "A_to_B",
                "ranking_idx": group_a,
                "evaluation_idx": group_b,
                "direction_offset": 0,
            },
            {
                "outer_rep": outer_rep,
                "outer_seed": outer_seed,
                "direction": "B_to_A",
                "ranking_idx": group_b,
                "evaluation_idx": group_a,
                "direction_offset": 1,
            },
        ])
    return rows


def load_compatible_legacy_csv(filename):
    frames = []
    sources = []
    for directory in LEGACY_OUTPUT_DIR_CANDIDATES:
        path = Path(directory) / filename
        if not path.exists() or path.stat().st_size == 0:
            continue
        try:
            frame = pd.read_csv(path)
        except Exception as exc:
            warnings.warn(f"Could not read {path}: {exc}")
            continue
        frame["_legacy_source_csv"] = str(path)
        frames.append(frame)
        sources.append(path)

    if not frames:
        return pd.DataFrame(), []

    combined = pd.concat(frames, ignore_index=True, sort=False)

    if "analysis_type" in combined.columns:
        combined = combined[
            combined["analysis_type"].astype(str) == ANALYSIS_TYPE
        ]
    if "fit_scope" in combined.columns:
        combined = combined[
            combined["fit_scope"].astype(str) == FIT_SCOPE
        ]

    return combined.reset_index(drop=True), sources


legacy_rank_raw, legacy_rank_sources = load_compatible_legacy_csv(
    "heldout_per_archetype_ranking_runs.csv"
)
legacy_run_raw, legacy_run_sources = load_compatible_legacy_csv(
    "heldout_topm_crossfit_runs.csv"
)
legacy_seed_raw, legacy_seed_sources = load_compatible_legacy_csv(
    "outer_split_seeds.csv"
)

print("Legacy ranking sources:", [str(p) for p in legacy_rank_sources])
print("Legacy run sources:", [str(p) for p in legacy_run_sources])
print("Legacy ranking rows:", len(legacy_rank_raw))
print("Legacy held-out rows:", len(legacy_run_raw))


RANK_CSV = OUTPUT_DIR / "temporal_crossfit_rankings.csv"
rank_df = load_csv(RANK_CSV)

RANK_KEYS = [
    "fit_fingerprint",
    "condition",
    "K",
    "outer_rep",
    "direction",
    "archetype",
]


def import_legacy_rankings_for_fit(
    fit,
    fit_label,
    fingerprint,
    condition,
):
    global rank_df

    if not REUSE_EXISTING_RANKINGS or len(legacy_rank_raw) == 0:
        return 0

    K = int(fit["K"])

    required = {
        "condition",
        "K",
        "outer_rep",
        "direction",
        "archetype",
    }
    if not required.issubset(legacy_rank_raw.columns):
        warnings.warn(
            "Legacy ranking CSV lacks required columns; rankings will be "
            "computed only where needed."
        )
        return 0

    current = legacy_rank_raw[
        (legacy_rank_raw["condition"].astype(str) == str(condition))
        & (legacy_rank_raw["K"].astype(int) == K)
    ].copy()

    if len(current) == 0:
        return 0

    rename = {}
    if "mean_accuracy" not in current.columns and "mean" in current.columns:
        rename["mean"] = "mean_accuracy"
    if rename:
        current = current.rename(columns=rename)

    for column, default in {
        "mean_accuracy": np.nan,
        "sem_accuracy": np.nan,
        "rank_mean": np.nan,
        "error_mean": np.nan,
        "outer_seed": np.nan,
        "n_ranking_subjects": 18,
    }.items():
        if column not in current.columns:
            current[column] = default

    current["fit_label"] = fit_label
    current["fit_fingerprint"] = fingerprint
    current["fit_path"] = str(fit["path"])

    # Verify or fill the expected split seeds.
    expected_seed = current.apply(
        lambda row: outer_seed_for(
            str(row["condition"]),
            int(row["outer_rep"]),
            int(row["K"]),
        ),
        axis=1,
    )

    if current["outer_seed"].notna().any():
        mismatch = (
            current["outer_seed"].notna()
            & (
                current["outer_seed"].astype("Int64")
                != pd.Series(expected_seed, index=current.index).astype("Int64")
            )
        )
        if mismatch.any():
            raise RuntimeError(
                "Existing ranking rows use a different outer-split seed "
                "scheme. Do not mix them with newly computed rows."
            )

    current["outer_seed"] = expected_seed.astype(int)

    keep = [
        "fit_label",
        "fit_fingerprint",
        "fit_path",
        "condition",
        "K",
        "outer_rep",
        "direction",
        "outer_seed",
        "n_ranking_subjects",
        "archetype",
        "mean_accuracy",
        "sem_accuracy",
        "rank_mean",
        "error_mean",
    ]

    current = current[keep]

    before = len(rank_df)
    rank_df = append_dedupe(
        rank_df,
        current,
        RANK_KEYS,
    )
    save_csv(rank_df, RANK_CSV)
    return len(rank_df) - before


def canonicalize_legacy_runs_for_fit(
    fit,
    fit_label,
    fingerprint,
    condition=None,
):
    if not REUSE_EXISTING_RECONSTRUCTION_RUNS or len(legacy_run_raw) == 0:
        return pd.DataFrame()

    K = int(fit["K"])

    required = {
        "condition",
        "K",
        "outer_rep",
        "direction",
        "top_m",
    }
    if not required.issubset(legacy_run_raw.columns):
        warnings.warn(
            "Legacy run CSV lacks required columns; held-out cells will be "
            "computed only where needed."
        )
        return pd.DataFrame()

    current = legacy_run_raw[
        legacy_run_raw["K"].astype(int) == K
    ].copy()

    if condition is not None:
        current = current[
            current["condition"].astype(str) == str(condition)
        ]

    if len(current) == 0:
        return pd.DataFrame()

    if "mean" not in current.columns and "accuracy" in current.columns:
        current = current.rename(columns={"accuracy": "mean"})

    defaults = {
        "err": np.nan,
        "std": np.nan,
        "count": np.nan,
        "rank_mean": np.nan,
        "error_mean": np.nan,
        "full_mean_same_eval": np.nan,
        "full_err_same_eval": np.nan,
        "chance": 1.0 / 300.0,
        "retention_above_chance": np.nan,
        "selected_archetypes": "[]",
        "outer_seed": np.nan,
        "n_ranking_subjects": 18,
        "n_evaluation_subjects": 18,
    }
    for column, default in defaults.items():
        if column not in current.columns:
            current[column] = default

    missing_retention = current["retention_above_chance"].isna()
    denominator = (
        current["full_mean_same_eval"].astype(float)
        - current["chance"].astype(float)
    )
    current.loc[
        missing_retention & denominator.abs().gt(1e-12),
        "retention_above_chance",
    ] = (
        current.loc[missing_retention, "mean"].astype(float)
        - current.loc[missing_retention, "chance"].astype(float)
    ) / denominator.loc[missing_retention]

    current["space"] = "reconstruction"
    current["fit_label"] = fit_label
    current["fit_fingerprint"] = fingerprint
    current["fit_path"] = str(fit["path"])

    expected_seed = current.apply(
        lambda row: outer_seed_for(
            str(row["condition"]),
            int(row["outer_rep"]),
            int(row["K"]),
        ),
        axis=1,
    )
    if current["outer_seed"].notna().any():
        mismatch = (
            current["outer_seed"].notna()
            & (
                current["outer_seed"].astype("Int64")
                != pd.Series(expected_seed, index=current.index).astype("Int64")
            )
        )
        if mismatch.any():
            raise RuntimeError(
                "Existing held-out rows use a different outer-split seed "
                "scheme. Do not mix them with newly computed rows."
            )
    current["outer_seed"] = expected_seed.astype(int)

    keep = [
        "space",
        "top_m",
        "selected_archetypes",
        "mean",
        "err",
        "std",
        "count",
        "rank_mean",
        "error_mean",
        "full_mean_same_eval",
        "full_err_same_eval",
        "retention_above_chance",
        "chance",
        "fit_label",
        "fit_fingerprint",
        "fit_path",
        "condition",
        "K",
        "outer_rep",
        "direction",
        "outer_seed",
        "n_ranking_subjects",
        "n_evaluation_subjects",
    ]
    return current[keep].reset_index(drop=True)


def ensure_rankings(
    fit,
    fit_label,
    fingerprint,
    condition,
):
    global rank_df

    K = int(fit["K"])
    subjects = condition_subjects_from_fit(fit, condition)
    definitions = direction_definitions(
        len(subjects),
        condition,
        K,
    )

    imported = import_legacy_rankings_for_fit(
        fit,
        fit_label,
        fingerprint,
        condition,
    )
    if imported:
        print(
            f"  imported {imported} existing ranking rows | "
            f"{condition} | K={K}",
            flush=True,
        )

    for archetype in range(K):
        existing = set()
        if len(rank_df):
            current = rank_df[
                (rank_df["fit_fingerprint"].astype(str) == str(fingerprint))
                & (rank_df["condition"].astype(str) == str(condition))
                & (rank_df["K"].astype(int) == K)
                & (rank_df["archetype"].astype(int) == int(archetype))
            ]
            existing = {
                (int(row.outer_rep), str(row.direction))
                for row in current.itertuples(index=False)
            }

        needed = [
            definition
            for definition in definitions
            if (
                int(definition["outer_rep"]),
                str(definition["direction"]),
            ) not in existing
        ]

        if not needed:
            continue

        U, gram = temporal_single_factors(
            subjects,
            archetype,
        )
        rows = []

        for definition in needed:
            summary = lowrank_single_decode_summary(
                U,
                gram,
                definition["ranking_idx"],
                nreps=NREPS_DECODE,
                seed=ranking_seed_for(
                    K,
                    definition["outer_rep"],
                    definition["direction_offset"],
                ),
            )
            rows.append({
                "fit_label": fit_label,
                "fit_fingerprint": fingerprint,
                "fit_path": str(fit["path"]),
                "condition": condition,
                "K": K,
                "outer_rep": int(definition["outer_rep"]),
                "direction": str(definition["direction"]),
                "outer_seed": int(definition["outer_seed"]),
                "n_ranking_subjects": int(len(definition["ranking_idx"])),
                "archetype": int(archetype),
                "mean_accuracy": summary["mean"],
                "sem_accuracy": summary["err"],
                "rank_mean": summary["rank_mean"],
                "error_mean": summary["error_mean"],
            })

        rank_df = append_dedupe(
            rank_df,
            pd.DataFrame(rows),
            RANK_KEYS,
        )

        if (archetype + 1) % 10 == 0 or archetype + 1 == K:
            save_csv(rank_df, RANK_CSV)
            print(
                f"  ranking {fit_label} | {condition} | "
                f"K={K} | {archetype + 1}/{K}",
                flush=True,
            )

        del U
        del gram
        gc.collect()

    save_csv(rank_df, RANK_CSV)
    return definitions, subjects


def ranked_for_direction(
    fingerprint,
    condition,
    K,
    outer_rep,
    direction,
):
    current = rank_df[
        (rank_df["fit_fingerprint"].astype(str) == str(fingerprint))
        & (rank_df["condition"].astype(str) == str(condition))
        & (rank_df["K"].astype(int) == int(K))
        & (rank_df["outer_rep"].astype(int) == int(outer_rep))
        & (rank_df["direction"].astype(str) == str(direction))
    ].copy()

    if current["archetype"].nunique() != int(K):
        raise RuntimeError(
            f"Incomplete ranking: {condition}, K={K}, rep={outer_rep}, "
            f"{direction}: {current['archetype'].nunique()}/{K}"
        )

    return (
        current
        .sort_values(
            ["mean_accuracy", "rank_mean", "error_mean", "archetype"],
            ascending=[False, False, True, True],
        )["archetype"]
        .astype(int)
        .tolist()
    )


## Matched retained fractions across temporal model order

For every requested model order, the notebook converts each retained fraction
to the nearest valid number of selected archetypes:

```python
m = round(fraction * K)
```

It first imports compatible exact `(K, m)` held-out cells from the established
temporal outputs. It also uses the stored full-model evaluation for `m = K`.
Only missing participant-held-out evaluations are decoded.


In [ ]:
# ============================================================
# MATCHED FRACTIONS — CANONICAL FIT, REUSE EXACT EXISTING CELLS
# ============================================================

FRACTION_RUN_CSV = OUTPUT_DIR / "matched_fraction_runs.csv"
FRACTION_RUN_KEYS = [
    "fit_fingerprint",
    "condition",
    "K",
    "outer_rep",
    "direction",
    "fraction_requested",
    "top_m",
]
fraction_run_df = load_csv(FRACTION_RUN_CSV)


def available_temporal_K():
    values = []
    for root in FIT_SEARCH_ROOTS:
        root = Path(root)
        if not root.exists():
            continue
        try:
            for path in root.glob(
                "**/temporalAA_across_acrossCond_K*.npz"
            ):
                match = re.search(r"_K(\d+)\.npz$", path.name)
                if match:
                    values.append(int(match.group(1)))
        except OSError:
            pass
    return sorted(set(values))


def prefill_fraction_rows_from_existing(
    fit,
    fit_label,
    fingerprint,
    mapping,
):
    existing_runs = canonicalize_legacy_runs_for_fit(
        fit,
        fit_label,
        fingerprint,
    )
    if len(existing_runs) == 0:
        return pd.DataFrame()

    rows = []
    K = int(fit["K"])

    group_columns = [
        "condition",
        "outer_rep",
        "direction",
    ]

    for keys, current in existing_runs.groupby(group_columns):
        condition, outer_rep, direction = keys

        for map_row in mapping.itertuples(index=False):
            requested = float(map_row.fraction_requested)
            top_m = int(map_row.top_m)

            exact = current[
                current["top_m"].astype(int) == top_m
            ]

            if len(exact):
                source = exact.iloc[0].to_dict()
            elif top_m == K and current["full_mean_same_eval"].notna().any():
                # The full model is already evaluated on these exact held-out
                # participants in every existing top-m row.
                base = current[
                    current["full_mean_same_eval"].notna()
                ].iloc[0].to_dict()
                source = base
                source["space"] = "reconstruction"
                source["top_m"] = K
                source["selected_archetypes"] = str(list(range(K)))
                source["mean"] = float(base["full_mean_same_eval"])
                source["err"] = float(base["full_err_same_eval"])
                source["retention_above_chance"] = 1.0
            else:
                continue

            source.update({
                "fraction_requested": requested,
                "actual_fraction": float(map_row.actual_fraction),
                "fit_label": fit_label,
                "fit_fingerprint": fingerprint,
                "fit_path": str(fit["path"]),
                "condition": str(condition),
                "K": K,
                "outer_rep": int(outer_rep),
                "direction": str(direction),
            })
            rows.append(source)

    return pd.DataFrame(rows)


if RUN_MATCHED_FRACTION:
    available = available_temporal_K()
    matched_K = (
        available
        if MATCHED_FRACTION_K_VALUES is None
        else [
            int(K)
            for K in MATCHED_FRACTION_K_VALUES
            if int(K) in available
        ]
    )
    print("Matched-fraction K values:", matched_K)

    for K in matched_K:
        fit_table = discover_fit_paths(K)
        if len(fit_table) == 0:
            continue

        fit_indices = [0]  # canonical/local only

        mapping = pd.DataFrame([
            {
                "fraction_requested": float(fraction),
                "top_m": int(
                    np.clip(
                        np.round(float(fraction) * int(K)),
                        1,
                        int(K),
                    )
                ),
            }
            for fraction in MATCHED_FRACTIONS
        ])
        mapping["actual_fraction"] = (
            mapping["top_m"] / float(K)
        )

        for fit_index in fit_indices:
            fit_row = fit_table.iloc[int(fit_index)]
            fit = load_fit(fit_row["fit_path"])
            fit_label = str(fit_row["fit_label"])
            fingerprint = str(fit_row["fingerprint"])

            prefilled = prefill_fraction_rows_from_existing(
                fit,
                fit_label,
                fingerprint,
                mapping,
            )
            if len(prefilled):
                before = len(fraction_run_df)
                fraction_run_df = append_dedupe(
                    fraction_run_df,
                    prefilled,
                    FRACTION_RUN_KEYS,
                )
                save_csv(fraction_run_df, FRACTION_RUN_CSV)
                print(
                    f"  K={K}: imported "
                    f"{len(fraction_run_df) - before} exact existing cells",
                    flush=True,
                )

            print("\n" + "=" * 88, flush=True)
            print(
                f"Matched fractions | {fit_label} | K={K}",
                flush=True,
            )
            print("=" * 88, flush=True)

            for condition in CONDITIONS:
                definitions, subjects = ensure_rankings(
                    fit,
                    fit_label,
                    fingerprint,
                    condition,
                )

                for definition in definitions:
                    outer_rep = int(definition["outer_rep"])
                    direction = str(definition["direction"])

                    expected = {
                        (
                            float(row.fraction_requested),
                            int(row.top_m),
                        )
                        for row in mapping.itertuples(index=False)
                    }
                    existing = set()

                    if len(fraction_run_df):
                        current = fraction_run_df[
                            (fraction_run_df["fit_fingerprint"].astype(str) == fingerprint)
                            & (fraction_run_df["condition"].astype(str) == condition)
                            & (fraction_run_df["K"].astype(int) == int(K))
                            & (fraction_run_df["outer_rep"].astype(int) == outer_rep)
                            & (fraction_run_df["direction"].astype(str) == direction)
                        ]
                        existing = {
                            (
                                float(row.fraction_requested),
                                int(row.top_m),
                            )
                            for row in current.itertuples(index=False)
                        }

                    missing = expected - existing
                    if (
                        not missing
                        and SKIP_EXISTING
                        and not FORCE_RECOMPUTE
                    ):
                        continue

                    ranked = ranked_for_direction(
                        fingerprint,
                        condition,
                        K,
                        outer_rep,
                        direction,
                    )
                    evaluation_subjects = [
                        subjects[int(index)]
                        for index in definition["evaluation_idx"]
                    ]
                    missing_m = sorted({
                        int(top_m)
                        for _, top_m in missing
                    })

                    evaluated = evaluate_reconstruction_topm_fast(
                        evaluation_subjects,
                        ranked,
                        missing_m,
                        NREPS_DECODE,
                        evaluation_seed_for(
                            K,
                            outer_rep,
                            definition["direction_offset"],
                        ),
                    )

                    rows = []
                    for map_row in mapping.itertuples(index=False):
                        pair = (
                            float(map_row.fraction_requested),
                            int(map_row.top_m),
                        )
                        if pair not in missing:
                            continue

                        source = evaluated[
                            evaluated["top_m"].astype(int)
                            == int(map_row.top_m)
                        ]
                        if len(source) != 1:
                            raise RuntimeError(
                                f"Missing K={K}, m={map_row.top_m}"
                            )

                        output = source.iloc[0].to_dict()
                        output.update({
                            "fit_label": fit_label,
                            "fit_fingerprint": fingerprint,
                            "fit_path": str(fit["path"]),
                            "condition": condition,
                            "K": int(K),
                            "outer_rep": outer_rep,
                            "direction": direction,
                            "outer_seed": int(definition["outer_seed"]),
                            "fraction_requested": float(
                                map_row.fraction_requested
                            ),
                            "actual_fraction": float(
                                map_row.actual_fraction
                            ),
                        })
                        rows.append(output)

                    fraction_run_df = append_dedupe(
                        fraction_run_df,
                        pd.DataFrame(rows),
                        FRACTION_RUN_KEYS,
                    )
                    save_csv(fraction_run_df, FRACTION_RUN_CSV)

                    print(
                        f"  {condition} | K={K} | outer "
                        f"{outer_rep + 1}/{N_OUTER_REPS} | "
                        f"{direction} | newly computed m={missing_m}",
                        flush=True,
                    )

                    del evaluation_subjects
                    gc.collect()

            del fit
            gc.collect()
else:
    print(
        "Matched-fraction analysis disabled. Enable it after validating "
        "the K=14 local output."
    )


In [ ]:
# ============================================================
# MATCHED FRACTION SUMMARY / PLOTS
# ============================================================

FRACTION_SUMMARY_CSV=OUTPUT_DIR/"matched_fraction_summary.csv"
fraction_run_df=load_csv(FRACTION_RUN_CSV)
if len(fraction_run_df):
    outer=(fraction_run_df.groupby(["fit_label","fit_fingerprint","condition","K","fraction_requested","top_m","actual_fraction","outer_rep"],as_index=False)
           .agg(accuracy=("mean","mean"),retention=("retention_above_chance","mean")))
    seed=(outer.groupby(["fit_label","fit_fingerprint","condition","K","fraction_requested","top_m","actual_fraction"],as_index=False)
          .agg(mean_accuracy=("accuracy","mean"),outer_sem_accuracy=("accuracy",sem),mean_retention=("retention","mean"),outer_sem_retention=("retention",sem)))
    fraction_summary=(seed.groupby(["condition","K","fraction_requested","top_m","actual_fraction"],as_index=False)
                      .agg(mean_accuracy=("mean_accuracy","mean"),fit_seed_sd_accuracy=("mean_accuracy","std"),mean_outer_sem_accuracy=("outer_sem_accuracy","mean"),mean_retention=("mean_retention","mean"),fit_seed_sd_retention=("mean_retention","std"),mean_outer_sem_retention=("outer_sem_retention","mean"),n_fit_seeds=("fit_label","nunique")))
    save_csv(fraction_summary,FRACTION_SUMMARY_CSV); display(fraction_summary.head(40))
    for condition in CONDITIONS:
        for fraction in MATCHED_FRACTIONS:
            cur=fraction_summary[(fraction_summary["condition"]==condition)&np.isclose(fraction_summary["fraction_requested"],float(fraction))].sort_values("K")
            if not len(cur): continue
            uncertainty=(cur["fit_seed_sd_retention"] if cur["n_fit_seeds"].max()>1 else cur["mean_outer_sem_retention"]).fillna(0)
            plt.figure(figsize=(7.5,4.5)); plt.plot(cur["K"],cur["mean_retention"],marker="o",linewidth=2)
            plt.fill_between(cur["K"],cur["mean_retention"]-uncertainty,cur["mean_retention"]+uncertainty,alpha=.2)
            plt.axhline(1.0,linestyle="--",linewidth=1); plt.xlabel("Temporal model order (K)"); plt.ylabel("Fraction of above-chance full-model performance retained")
            plt.title(f"Temporal MS-AA | {condition} | requested fraction={fraction:.2f}"); plt.tight_layout()
            plt.savefig(OUTPUT_DIR/f"matched_fraction_{condition}_{fraction:.2f}.png",dpi=300,bbox_inches="tight"); plt.show()
else:
    fraction_summary=pd.DataFrame(); print("No matched-fraction results yet.")


In [ ]:
# ============================================================
# MATCHED-FRACTION PEAK / COMPACT-SCALE SUMMARY
# ============================================================

MATCHED_FRACTION_PEAK_CSV = (
    OUTPUT_DIR / "matched_fraction_peak_by_condition.csv"
)

if "fraction_summary" in globals() and len(fraction_summary):
    peak_rows = []

    for (condition, fraction), current in fraction_summary.groupby(
        ["condition", "fraction_requested"]
    ):
        current = current.dropna(subset=["mean_retention"]).copy()
        if len(current) == 0:
            continue

        best = current.loc[current["mean_retention"].idxmax()]

        peak_rows.append({
            "condition": str(condition),
            "fraction_requested": float(fraction),
            "best_K": int(best["K"]),
            "best_top_m": int(best["top_m"]),
            "best_actual_fraction": float(best["actual_fraction"]),
            "best_mean_retention": float(best["mean_retention"]),
            "K14_mean_retention": (
                float(
                    current.loc[
                        current["K"].astype(int) == REFERENCE_K,
                        "mean_retention",
                    ].iloc[0]
                )
                if (current["K"].astype(int) == REFERENCE_K).any()
                else np.nan
            ),
        })

    matched_fraction_peak_df = pd.DataFrame(peak_rows)
    save_csv(matched_fraction_peak_df, MATCHED_FRACTION_PEAK_CSV)
    display(matched_fraction_peak_df)
else:
    matched_fraction_peak_df = pd.DataFrame()
    print("No matched-fraction summary is available yet.")


## How to interpret the result

The primary output is:

```text
outputs/temporal_compact_dictionary_K14_local/
matched_fraction_summary.csv
```

Focus first on the 50% and 70% retention plots for intact and word-scrambled
listening.

- If retention remains unusually high around K = 12--16 at a fixed fraction,
  that supports a favorable compact temporal representational regime.
- If the low-K advantage largely disappears, the fixed-top-10 peak is mainly a
  retained-fraction effect: ten archetypes perform well at K = 14 because they
  comprise most of that compact basis.
- The 100% curve describes full-model performance and is not a sparsity test.

Run from the repository root:

```bash
python scripts/execute_notebook.py \
  analysis/03_core_results/revision_10d_temporal_matched_fraction_K5_K50.ipynb
```

The notebook is resumable with:

```python
SKIP_EXISTING = True
FORCE_RECOMPUTE = False
```
